## Weather Summarizer

In [ ]:
import os
from dotenv import load_dotenv
from tavily import TavilyClient
from IPython.display import Markdown, display   # for displaying the final report nicely  
from openai import OpenAI  # for calling Gemini, since you're using GOOGLE_API_KEY

load_dotenv(override=True)
tavily_api_key = os.getenv('TAVILY_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

tavily = TavilyClient(api_key=tavily_api_key)
query = "Give me a summary of the last 10 days weather in New Delhi City"

summarize_system_prompt = """
You are a research assistant that reads content from multiple webpages about a topic
and writes ONE clear, well-organized summary answering the topic. Ignore navigation
text, ads, and boilerplate. Ignore sources that are irrelevant to the topic rather
than including noise. Where useful, mention which source a fact came from.
"""

summarize_user_prompt_template = """
Topic: {topic}

Below are excerpts from multiple webpages:

{sources_block}

Write one combined summary answering the topic above, using only the relevant sources.
"""

def search(query):
    # 1. Search for the topic using Tavily
    result = tavily.search(query=query)
    # 2. Get the content from the first result
    source = []
    for item in result["results"]:
        source.append({
            "title": item["title"],
            "url": item["url"],
            "content": item["content"]
        })
    return source

def build_sources_block(sources):
    blocks = []
    for item in sources:
        block = f"Source: {item['url']}\nTitle: {item['title']}\nContent: {item['content']}"
        blocks.append(block)
    return "\n\n".join(blocks)


def summarize_all(topic, sources):
    sources_block = build_sources_block(sources)

    gemini = OpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=google_api_key
    )
    response = gemini.chat.completions.create(
        model="gemini-3.6-flash",
        messages=[
            {"role": "system", "content": summarize_system_prompt},
            {"role": "user", "content": summarize_user_prompt_template.format(topic=topic, sources_block=sources_block)}
        ]
    )
    display(Markdown(f"### Summary:\n{response.choices[0].message.content}"))


summarize_all(query, search(query))



## User input query

In [ ]:
import os
from dotenv import load_dotenv
from tavily import TavilyClient
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
tavily_api_key = os.getenv('TAVILY_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

tavily = TavilyClient(api_key=tavily_api_key)

query = input("Enter your query :")

summarize_system_prompt = """
You are a research assistant that reads content from multiple webpages about a topic
and writes ONE clear, well-organized summary answering the topic. Ignore navigation
text, ads, and boilerplate. Ignore sources that are irrelevant to the topic rather
than including noise. Where useful, mention which source a fact came from.
"""

summarize_user_prompt_template = """
Topic: {topic}

Below are excerpts from multiple webpages:

{sources_block}

Write one combined summary answering the topic above, using only the relevant sources.
"""
def search(query):
    # 1. Search for the topic using Tavily
    result = tavily.search(query=query)
    
    # 2. Get the content from the first result
    source = []
    for item in result["results"]:
        source.append({
            "title": item["title"],
            "url": item["url"],
            "content": item["content"]
        })
    return source


def build_sources_block(sources):
    blocks = []
    for item in sources:
        block = f"Source: {item['url']}\nTitle: {item['title']}\nContent: {item['content']}"
        blocks.append(block)
    return "\n\n".join(blocks)

def summarize_all(topic, sources):
    sources_block = build_sources_block(sources)

    gemini = OpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=google_api_key
    )
    response = gemini.chat.completions.create(
        model="gemini-3.6-flash",
        messages=[
            {"role": "system", "content": summarize_system_prompt},
            {"role": "user", "content": summarize_user_prompt_template.format(topic=topic, sources_block=sources_block)}
        ]
    )
    display(Markdown(f"### Summary:\n{response.choices[0].message.content}"))


summarize_all(query, search(query))



## web page content Summarizer

In [ ]:
import os
from dotenv import load_dotenv
from tavily import TavilyClient
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
tavily_api_key = os.getenv('TAVILY_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

tavily = TavilyClient(api_key=tavily_api_key)
gemini = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=google_api_key
)

webpage_summary_system_prompt = """
You are an assistant that summarizes the content of a single webpage clearly.
Ignore navigation text, ads, and boilerplate. Focus only on the main content and key points.
Respond in a well-organized paragraphs or bullet points, whichever fits the content best.
"""

webpage_summary_user_prompt_template = """
Please summarize the following webpage content (source: {url}):

{content}
"""

def summarize_all(URL, Content):
    response = gemini.chat.completions.create(
        model="gemini-3.6-flash",
        messages=[
            {"role": "system", "content": webpage_summary_system_prompt},
            {"role": "user", "content": webpage_summary_user_prompt_template.format(url=URL, content=Content)}
        ]
    )
    display(Markdown(f"### Summary:\n{response.choices[0].message.content}"))

URL = str(input("Enter the complete url : "))
response = tavily.extract(urls=URL)
page_content = response['results'][0]['raw_content']

summarize_all(URL, page_content)